In [1]:
"""
ORB (Opening Range Breakout) Bot — SPY / QQQ
Replicates the TradingView ORB Strategy [LuciTech] using IBKR paper trading.

Architecture:
  - Single asyncio event loop shared by ib_insync and the scheduler
  - 5-min bar polling via reqHistoricalDataAsync (useRTH=False)
  - ORB window: 9:30–9:45 ET  |  Trade cutoff: 11:00 ET
  - Stop types: ATR | Candle | Midline
  - Breakeven trailing, position sizing by risk %
  - All IB calls are async coroutines (ib_insync 0.9.86 compatible)
"""

import asyncio
import sys
import logging
import random
from datetime import datetime, timezone, time as dtime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from pymongo import MongoClient

from ib_insync import IB, Stock, LimitOrder, StopOrder, BracketOrder, util

import warnings
warnings.filterwarnings("ignore")

ET = ZoneInfo("America/New_York")


# ══════════════════════════════════════════════════════════════════════════════
# Logging — UTF-8 safe (works in Jupyter + Windows cp1252)
# ══════════════════════════════════════════════════════════════════════════════
class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)
    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("orb_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("orb_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# Configuration  — edit these
# ══════════════════════════════════════════════════════════════════════════════
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497                   # TWS paper=7497 | IB Gateway paper=4002
IBKR_CLIENT_ID = random.randint(1, 999)

SYMBOLS        = ["SPY", "QQQ"]        # symbols to trade
BAR_SIZE       = "5 mins"              # IBKR bar size string
POLL_SECONDS   = 60                    # how often to re-check bars (seconds)

# ORB session window (ET)
ORB_START      = dtime(9, 30)
ORB_END        = dtime(9, 45)
TRADE_CUTOFF   = dtime(11, 0)
EOD_CLOSE      = dtime(15, 55)         # force-close all positions before close

# Risk management (mirrors TV inputs)
RISK_PERCENT   = 0.02                  # 2% of equity per trade
RISK_REWARD    = 2.0                   # RR ratio
BREAKEVEN_R    = 1.0                   # move SL to BE after 1R (0 = disabled)
SL_TYPE        = "ATR"                 # "ATR" | "Candle" | "Midline"
ATR_LENGTH     = 14
ATR_MULTI      = 1.0                   # ATR multiplier for SL distance
CANDLE_LENGTH  = 0                     # bars back for candle SL (0 = last bar)

# Relative volume filter
USE_RVOL       = False
RVOL_LENGTH    = 20
RVOL_MIN       = 1.5

# Position filter
ALLOW_LONG     = True
ALLOW_SHORT    = True

# Time filter (ET) — None = disabled
TIME_FILTER_START = None               # e.g. dtime(9, 30)
TIME_FILTER_END   = None               # e.g. dtime(11, 0)

# Day filter — None = all days
ALLOWED_DAYS   = None                  # e.g. {0,1,2,3,4} = Mon-Fri (0=Mon)

# MongoDB
MONGO_URI      = "mongodb://localhost:27017/"
DB_NAME        = "orb_bot"


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB
# ══════════════════════════════════════════════════════════════════════════════
_mongo         = MongoClient(MONGO_URI)
_db            = _mongo[DB_NAME]
trades_col     = _db["trades"]
signals_col    = _db["signals"]


def save_trade_db(doc: dict):
    trades_col.insert_one(doc)
    log.info(f"DB: trade saved  {doc['symbol']}  side={doc['side']}  entry={doc['entry_price']}")


def update_trade_db(symbol: str, update: dict):
    trades_col.update_one({"symbol": symbol, "status": "open"}, {"$set": update})


def get_open_trade(symbol: str) -> dict | None:
    return trades_col.find_one({"symbol": symbol, "status": "open"})


# ══════════════════════════════════════════════════════════════════════════════
# Indicators
# ══════════════════════════════════════════════════════════════════════════════
def calc_atr(df: pd.DataFrame, length: int = 14) -> pd.Series:
    """True Range → SMA-smoothed ATR (mirrors TV SMA smoothing default)."""
    h, l, c = df["high"], df["low"], df["close"]
    prev_c  = c.shift(1)
    tr      = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    return tr.rolling(length).mean()


def calc_rvol(df: pd.DataFrame, length: int = 20) -> float:
    """Relative volume of the last bar vs rolling average."""
    if len(df) < length + 1:
        return 1.0
    avg = df["volume"].iloc[-(length+1):-1].mean()
    if avg == 0:
        return 1.0
    return float(df["volume"].iloc[-1] / avg)


# ══════════════════════════════════════════════════════════════════════════════
# ORB State  (one instance per symbol)
# ══════════════════════════════════════════════════════════════════════════════
class ORBState:
    def __init__(self, symbol: str):
        self.symbol          = symbol
        self.date            = None       # current trading date (ET)
        self.orb_high        = None
        self.orb_low         = None
        self.orb_mid         = None
        self.orb_finalized   = False      # True once 9:45 bar is complete
        self.bull_fired      = False      # breakout signals fire once per day
        self.bear_fired      = False
        # Active trade state
        self.in_trade        = False
        self.side            = None       # "LONG" | "SHORT"
        self.entry_price     = None
        self.stop_level      = None
        self.target_level    = None
        self.initial_stop    = None
        self.moved_to_be     = False
        self.order_id        = None
        self.qty             = 0

    def reset_day(self, date):
        """Called at the start of each new trading day."""
        self.date          = date
        self.orb_high      = None
        self.orb_low       = None
        self.orb_mid       = None
        self.orb_finalized = False
        self.bull_fired    = False
        self.bear_fired    = False
        log.info(f"{self.symbol}: day reset for {date}")

    def update_orb(self, bar_high: float, bar_low: float):
        """Expand ORB range with a new bar inside the 9:30-9:45 window."""
        if self.orb_high is None:
            self.orb_high = bar_high
            self.orb_low  = bar_low
        else:
            self.orb_high = max(self.orb_high, bar_high)
            self.orb_low  = min(self.orb_low,  bar_low)
        self.orb_mid = (self.orb_high + self.orb_low) / 2

    def orb_ready(self) -> bool:
        return (self.orb_finalized
                and self.orb_high is not None
                and self.orb_low  is not None)


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Async Client
# ══════════════════════════════════════════════════════════════════════════════
class IBKRClient:
    def __init__(self):
        self.ib = IB()

    async def connect(self):
        await self.ib.connectAsync(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    async def get_equity(self) -> float:
        """Return net liquidation value of the account."""
        account_values = await self.ib.accountSummaryAsync()
        for av in account_values:
            if av.tag == "NetLiquidation" and av.currency == "USD":
                return float(av.value)
        # Fallback: use reqAccountValues
        vals = self.ib.accountValues()
        for v in vals:
            if v.tag == "NetLiquidation" and v.currency == "USD":
                return float(v.value)
        log.warning("Could not fetch equity; defaulting to 100000")
        return 100_000.0

    async def get_bars(self, symbol: str, duration: str = "2 D") -> pd.DataFrame | None:
        """Fetch 5-min OHLCV bars including pre/post market."""
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        for what in ("TRADES", "MIDPOINT"):
            raw = await self.ib.reqHistoricalDataAsync(
                contract,
                endDateTime="",
                durationStr=duration,
                barSizeSetting=BAR_SIZE,
                whatToShow=what,
                useRTH=False,
            )
            if raw:
                df = pd.DataFrame([{
                    "ts":     b.date,
                    "open":   b.open,
                    "high":   b.high,
                    "low":    b.low,
                    "close":  b.close,
                    "volume": b.volume,
                } for b in raw])
                # Ensure tz-aware datetime index in ET
                df["ts"] = pd.to_datetime(df["ts"], utc=True).dt.tz_convert(ET)
                df.set_index("ts", inplace=True)
                log.debug(f"{symbol}: {len(df)} bars ({what})  last={df.index[-1]}")
                return df
        log.warning(f"{symbol}: no bars returned")
        return None

    async def get_last_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        await asyncio.sleep(3)
        self.ib.cancelMktData(contract)
        for val in [ticker.last, ticker.ask, ticker.close]:
            if val and float(val) > 0:
                return float(val)
        return None

    def has_position(self, symbol: str) -> bool:
        return any(
            p.contract.symbol == symbol and p.position != 0
            for p in self.ib.positions()
        )

    async def place_bracket(
        self, symbol: str, side: str,
        qty: int, entry: float, stop: float, target: float
    ):
        """Place a limit entry with stop-loss and take-profit bracket."""
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)

        action     = "BUY" if side == "LONG" else "SELL"
        sl_action  = "SELL" if side == "LONG" else "BUY"

        entry_order = LimitOrder(action, qty, round(entry, 2))
        entry_order.outsideRth = True
        entry_order.tif        = "DAY"

        sl_order    = StopOrder(sl_action, qty, round(stop, 2))
        sl_order.outsideRth = True
        sl_order.tif        = "DAY"

        tp_order    = LimitOrder(sl_action, qty, round(target, 2))
        tp_order.outsideRth = True
        tp_order.tif        = "DAY"

        bracket = self.ib.bracketOrder(action, qty,
                                        limitPrice=round(entry, 2),
                                        takeProfitPrice=round(target, 2),
                                        stopLossPrice=round(stop, 2))
        for o in bracket:
            o.outsideRth = True
            o.tif        = "DAY"

        trades = []
        for o in bracket:
            t = self.ib.placeOrder(contract, o)
            trades.append(t)
        await asyncio.sleep(1)

        parent_id = trades[0].order.orderId
        log.info(
            f"BRACKET {side} {symbol} x{qty}  entry={entry:.2f}  "
            f"stop={stop:.2f}  target={target:.2f}  parentId={parent_id}"
        )
        return trades

    async def cancel_all_orders(self, symbol: str):
        for trade in self.ib.openTrades():
            if trade.contract.symbol == symbol:
                self.ib.cancelOrder(trade.order)
                log.info(f"Cancelled order {trade.order.orderId} for {symbol}")
        await asyncio.sleep(0.5)

    async def close_position(self, symbol: str, qty: int, side: str):
        """Market-close an open position."""
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        action = "SELL" if side == "LONG" else "BUY"
        from ib_insync import MarketOrder
        order = MarketOrder(action, abs(qty))
        order.outsideRth = True
        trade = self.ib.placeOrder(contract, order)
        await asyncio.sleep(1)
        log.info(f"CLOSE {symbol} {action} x{abs(qty)}  orderId={trade.order.orderId}")
        return trade

    async def modify_stop(self, symbol: str, side: str,
                          qty: int, new_stop: float, old_stop_order_id: int):
        """Cancel existing stop and place a new one at breakeven."""
        await self.cancel_all_orders(symbol)
        await asyncio.sleep(0.3)
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        action = "SELL" if side == "LONG" else "BUY"
        order  = StopOrder(action, qty, round(new_stop, 2))
        order.outsideRth = True
        order.tif        = "DAY"
        trade  = self.ib.placeOrder(contract, order)
        await asyncio.sleep(0.5)
        log.info(f"STOP MODIFIED {symbol} -> {new_stop:.2f}")
        return trade


# ══════════════════════════════════════════════════════════════════════════════
# Signal + Risk Logic
# ══════════════════════════════════════════════════════════════════════════════
def calc_stop_level(df: pd.DataFrame, side: str, state: ORBState,
                    last_close: float, atr: float) -> float:
    """Mirror TV's stop-loss calculation."""
    if SL_TYPE == "ATR":
        if side == "LONG":
            return last_close - atr * ATR_MULTI
        else:
            return last_close + atr * ATR_MULTI
    elif SL_TYPE == "Candle":
        idx = -(CANDLE_LENGTH + 1)
        if side == "LONG":
            return float(df["low"].iloc[idx])
        else:
            return float(df["high"].iloc[idx])
    else:  # Midline
        return state.orb_mid


def calc_position_size(equity: float, entry: float, stop: float) -> int:
    """Risk-based position sizing, same formula as TV strategy."""
    stop_distance = abs(entry - stop)
    if stop_distance <= 0:
        return 0
    risk_amount   = equity * RISK_PERCENT
    raw_size      = risk_amount / stop_distance
    return max(1, int(raw_size))


def passes_filters(now_et: datetime, side: str, rvol: float) -> bool:
    """Apply time, day, position-type, and RVOL filters."""
    # Time filter
    if TIME_FILTER_START and TIME_FILTER_END:
        t = now_et.time()
        if not (TIME_FILTER_START <= t <= TIME_FILTER_END):
            return False
    # Day filter
    if ALLOWED_DAYS is not None:
        if now_et.weekday() not in ALLOWED_DAYS:
            return False
    # Position type
    if side == "LONG"  and not ALLOW_LONG:  return False
    if side == "SHORT" and not ALLOW_SHORT: return False
    # RVOL
    if USE_RVOL and rvol < RVOL_MIN:
        log.info(f"RVOL filter failed: {rvol:.2f} < {RVOL_MIN}")
        return False
    return True


# ══════════════════════════════════════════════════════════════════════════════
# Per-Symbol Processing Loop
# ══════════════════════════════════════════════════════════════════════════════
async def process_symbol(symbol: str, state: ORBState, ibkr: IBKRClient):
    """Called every POLL_SECONDS. Implements the full ORB state machine."""
    now_et = datetime.now(ET)
    today  = now_et.date()

    # ── Day reset ────────────────────────────────────────────────────────────
    if state.date != today:
        state.reset_day(today)

    # ── Fetch bars ───────────────────────────────────────────────────────────
    df = await ibkr.get_bars(symbol)
    if df is None or len(df) < ATR_LENGTH + 2:
        log.warning(f"{symbol}: not enough bars")
        return

    # ── EOD close ────────────────────────────────────────────────────────────
    if now_et.time() >= EOD_CLOSE and state.in_trade:
        log.info(f"{symbol}: EOD -- closing position")
        await ibkr.cancel_all_orders(symbol)
        await ibkr.close_position(symbol, state.qty, state.side)
        update_trade_db(symbol, {
            "status":      "closed",
            "exit_reason": "EOD",
            "exited_at":   datetime.now(timezone.utc),
        })
        state.in_trade = False
        return

    # Past cutoff — no new entries
    past_cutoff = now_et.time() >= TRADE_CUTOFF

    # ── Build ORB from bars in window ─────────────────────────────────────────
    orb_bars = df.between_time(ORB_START.strftime("%H:%M"),
                               ORB_END.strftime("%H:%M"))
    if not orb_bars.empty:
        for _, row in orb_bars.iterrows():
            state.update_orb(float(row["high"]), float(row["low"]))

    # Finalize ORB once we're past 9:45
    if now_et.time() >= ORB_END and not state.orb_finalized:
        if state.orb_high is not None:
            state.orb_finalized = True
            log.info(
                f"{symbol}: ORB finalized  high={state.orb_high:.2f}  "
                f"low={state.orb_low:.2f}  mid={state.orb_mid:.2f}"
            )

    if not state.orb_ready():
        log.debug(f"{symbol}: ORB not ready yet")
        return

    # ── Current bar values ───────────────────────────────────────────────────
    last_bar   = df.iloc[-1]
    prev_bar   = df.iloc[-2]
    last_close = float(last_bar["close"])
    prev_close = float(prev_bar["close"])

    atr_series = calc_atr(df, ATR_LENGTH)
    atr        = float(atr_series.iloc[-1])
    rvol       = calc_rvol(df, RVOL_LENGTH)

    log.info(
        f"{symbol}: close={last_close:.2f}  ORB[{state.orb_low:.2f}-{state.orb_high:.2f}]"
        f"  ATR={atr:.4f}  RVOL={rvol:.2f}"
    )

    # ── Breakeven check on open trade ────────────────────────────────────────
    if state.in_trade and BREAKEVEN_R > 0 and not state.moved_to_be:
        init_risk = abs(state.entry_price - state.initial_stop)
        if state.side == "LONG":
            profit = float(df["high"].iloc[-1]) - state.entry_price
        else:
            profit = state.entry_price - float(df["low"].iloc[-1])

        if profit >= BREAKEVEN_R * init_risk:
            log.info(f"{symbol}: BREAKEVEN triggered at {state.entry_price:.2f}")
            state.moved_to_be = True
            state.stop_level  = state.entry_price
            await ibkr.modify_stop(symbol, state.side, state.qty,
                                   state.entry_price, -1)
            update_trade_db(symbol, {"stop_level": state.entry_price,
                                     "moved_to_be": True})

    # ── No new entries if already in trade or past cutoff ────────────────────
    if state.in_trade or past_cutoff:
        return

    # ── Breakout detection (mirrors TV close > sessionHigh logic) ───────────
    bull_breakout = (
        not state.bull_fired
        and last_close > state.orb_high
        and prev_close <= state.orb_high
    )
    bear_breakout = (
        not state.bear_fired
        and last_close < state.orb_low
        and prev_close >= state.orb_low
    )

    if not bull_breakout and not bear_breakout:
        return

    side = "LONG" if bull_breakout else "SHORT"

    if bull_breakout:
        state.bull_fired = True
        log.info(f"{symbol}: BULL BREAKOUT  close={last_close:.2f} > ORB_HIGH={state.orb_high:.2f}")
    else:
        state.bear_fired = True
        log.info(f"{symbol}: BEAR BREAKOUT  close={last_close:.2f} < ORB_LOW={state.orb_low:.2f}")

    # ── Filters ──────────────────────────────────────────────────────────────
    if not passes_filters(now_et, side, rvol):
        log.info(f"{symbol}: breakout filtered out  side={side}")
        return

    # ── Risk calculations ────────────────────────────────────────────────────
    entry = last_close
    stop  = calc_stop_level(df, side, state, last_close, atr)

    if side == "LONG"  and stop >= entry:
        log.warning(f"{symbol}: LONG stop {stop:.2f} >= entry {entry:.2f} -- skipping")
        return
    if side == "SHORT" and stop <= entry:
        log.warning(f"{symbol}: SHORT stop {stop:.2f} <= entry {entry:.2f} -- skipping")
        return

    stop_dist = abs(entry - stop)
    if side == "LONG":
        target = entry + RISK_REWARD * stop_dist
    else:
        target = entry - RISK_REWARD * stop_dist

    equity = await ibkr.get_equity()
    qty    = calc_position_size(equity, entry, stop)
    if qty <= 0:
        log.warning(f"{symbol}: position size 0 -- skipping")
        return

    log.info(
        f"{symbol}: {side}  entry={entry:.2f}  stop={stop:.2f}  "
        f"target={target:.2f}  qty={qty}  equity={equity:.0f}"
    )

    # ── Place bracket order ───────────────────────────────────────────────────
    trades = await ibkr.place_bracket(symbol, side, qty, entry, stop, target)

    # ── Update state ──────────────────────────────────────────────────────────
    state.in_trade     = True
    state.side         = side
    state.entry_price  = entry
    state.stop_level   = stop
    state.target_level = target
    state.initial_stop = stop
    state.moved_to_be  = False
    state.qty          = qty
    state.order_id     = trades[0].order.orderId

    # ── Persist to MongoDB ────────────────────────────────────────────────────
    save_trade_db({
        "symbol":        symbol,
        "side":          side,
        "entry_price":   entry,
        "stop_level":    stop,
        "target_level":  target,
        "initial_stop":  stop,
        "qty":           qty,
        "order_id":      state.order_id,
        "atr":           atr,
        "rvol":          rvol,
        "orb_high":      state.orb_high,
        "orb_low":       state.orb_low,
        "orb_mid":       state.orb_mid,
        "status":        "open",
        "moved_to_be":   False,
        "entered_at":    datetime.now(timezone.utc),
        "exit_price":    None,
        "exit_reason":   None,
        "exited_at":     None,
        "pnl_pct":       None,
    })

    # ── Log signal ────────────────────────────────────────────────────────────
    signals_col.insert_one({
        "symbol":    symbol,
        "side":      side,
        "entry":     entry,
        "stop":      stop,
        "target":    target,
        "rvol":      rvol,
        "timestamp": datetime.now(timezone.utc),
    })


# ══════════════════════════════════════════════════════════════════════════════
# Main Event Loop
# ══════════════════════════════════════════════════════════════════════════════
async def main():
    log.info("=== ORB Bot Starting ===")
    log.info(f"Symbols: {SYMBOLS}  |  Bar: {BAR_SIZE}  |  Poll: {POLL_SECONDS}s")
    log.info(f"ORB window: {ORB_START}-{ORB_END} ET  |  Cutoff: {TRADE_CUTOFF} ET")
    log.info(f"SL: {SL_TYPE}  RR: {RISK_REWARD}  Risk: {RISK_PERCENT*100:.1f}%  BE@{BREAKEVEN_R}R")

    util.patchAsyncio()

    ibkr   = IBKRClient()
    await ibkr.connect()

    states = {sym: ORBState(sym) for sym in SYMBOLS}

    try:
        while True:
            for symbol in SYMBOLS:
                try:
                    await process_symbol(symbol, states[symbol], ibkr)
                except Exception as e:
                    log.error(f"{symbol}: error in process_symbol - {e}", exc_info=True)

            log.info(f"Sleeping {POLL_SECONDS}s...")
            await asyncio.sleep(POLL_SECONDS)

    except KeyboardInterrupt:
        log.info("ORB Bot stopped by user.")
    finally:
        # Cancel all open orders and close positions on shutdown
        for symbol in SYMBOLS:
            state = states[symbol]
            if state.in_trade:
                log.info(f"{symbol}: shutdown -- closing position")
                try:
                    await ibkr.cancel_all_orders(symbol)
                    await ibkr.close_position(symbol, state.qty, state.side)
                except Exception as e:
                    log.error(f"{symbol}: shutdown close error - {e}")
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    if loop.is_running():
        # Jupyter / already-running loop — schedule as a task
        loop.create_task(main())
    else:
        util.run(main())

2026-04-03 07:45:39,894 [INFO] === ORB Bot Starting ===
2026-04-03 07:45:39,896 [INFO] Symbols: ['SPY', 'QQQ']  |  Bar: 5 mins  |  Poll: 60s
2026-04-03 07:45:39,897 [INFO] ORB window: 09:30:00-09:45:00 ET  |  Cutoff: 11:00:00 ET
2026-04-03 07:45:39,898 [INFO] SL: ATR  RR: 2.0  Risk: 2.0%  BE@1.0R


Error 326, reqId -1: Unable to connect as the client id is already in use. Retry with a unique client id.
Peer closed connection. clientId 863 already in use?
API connection failed: TimeoutError()


In [1]:
"""
ORB (Opening Range Breakout) Bot — SPY / QQQ
Replicates the TradingView ORB Strategy [LuciTech] using IBKR paper trading.

Architecture:
  - Single asyncio event loop shared by ib_insync and the scheduler
  - 5-min bar polling via reqHistoricalDataAsync (useRTH=False)
  - ORB window: 9:30–9:45 ET  |  Trade cutoff: 11:00 ET
  - Stop types: ATR | Candle | Midline
  - Breakeven trailing, fixed quantity of 10 shares per trade
  - All IB calls are async coroutines (ib_insync 0.9.86 compatible)
"""

import asyncio
import sys
import logging
import random
from datetime import datetime, timezone, time as dtime
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from pymongo import MongoClient

from ib_insync import IB, Stock, LimitOrder, StopOrder, MarketOrder, util

import warnings
warnings.filterwarnings("ignore")

ET = ZoneInfo("America/New_York")


# ══════════════════════════════════════════════════════════════════════════════
# Logging — UTF-8 safe (works in Jupyter + Windows cp1252)
# ══════════════════════════════════════════════════════════════════════════════
class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try:
            super().emit(record)
        except UnicodeEncodeError:
            record.msg  = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try:
                super().emit(record)
            except Exception:
                self.handleError(record)
    def _try_utf8(self):
        try:
            if hasattr(self.stream, "reconfigure"):
                self.stream.reconfigure(encoding="utf-8", errors="replace")
        except Exception:
            pass

def _make_logger() -> logging.Logger:
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh  = _Utf8SafeStreamHandler(sys.stdout)
    sh._try_utf8()
    sh.setFormatter(fmt)
    fh  = logging.FileHandler("orb_bot.log", encoding="utf-8")
    fh.setFormatter(fmt)
    logger = logging.getLogger("orb_bot")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger

log = _make_logger()


# ══════════════════════════════════════════════════════════════════════════════
# Configuration  — edit these
# ══════════════════════════════════════════════════════════════════════════════
IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497                   # TWS paper=7497 | IB Gateway paper=4002
IBKR_CLIENT_ID = random.randint(1, 999)

SYMBOLS        = ["SPY", "QQQ"]        # symbols to trade
BAR_SIZE       = "5 mins"              # IBKR bar size string
POLL_SECONDS   = 60                    # how often to re-check bars (seconds)

# ── Fixed position size ───────────────────────────────────────────────────────
FIXED_QTY      = 10                    # always buy/sell exactly 10 shares

# ORB session window (ET)
ORB_START      = dtime(9, 30)
ORB_END        = dtime(9, 45)
TRADE_CUTOFF   = dtime(11, 0)
EOD_CLOSE      = dtime(15, 55)         # force-close all positions before close

# Risk management (mirrors TV inputs — RISK_PERCENT / get_equity no longer
# used for sizing but kept for reference / future re-enablement)
RISK_REWARD    = 2.0                   # RR ratio for target calculation
BREAKEVEN_R    = 1.0                   # move SL to BE after 1R (0 = disabled)
SL_TYPE        = "ATR"                 # "ATR" | "Candle" | "Midline"
ATR_LENGTH     = 14
ATR_MULTI      = 1.0                   # ATR multiplier for SL distance
CANDLE_LENGTH  = 0                     # bars back for candle SL (0 = last bar)

# Relative volume filter
USE_RVOL       = False
RVOL_LENGTH    = 20
RVOL_MIN       = 1.5

# Position filter
ALLOW_LONG     = True
ALLOW_SHORT    = True

# Time filter (ET) — None = disabled
TIME_FILTER_START = None               # e.g. dtime(9, 30)
TIME_FILTER_END   = None               # e.g. dtime(11, 0)

# Day filter — None = all days
ALLOWED_DAYS   = None                  # e.g. {0,1,2,3,4} = Mon-Fri (0=Mon)

# MongoDB
MONGO_URI      = "mongodb://localhost:27017/"
DB_NAME        = "orb_bot"


# ══════════════════════════════════════════════════════════════════════════════
# MongoDB
# ══════════════════════════════════════════════════════════════════════════════
_mongo         = MongoClient(MONGO_URI)
_db            = _mongo[DB_NAME]
trades_col     = _db["trades"]
signals_col    = _db["signals"]


def save_trade_db(doc: dict):
    trades_col.insert_one(doc)
    log.info(f"DB: trade saved  {doc['symbol']}  side={doc['side']}  entry={doc['entry_price']}")


def update_trade_db(symbol: str, update: dict):
    trades_col.update_one({"symbol": symbol, "status": "open"}, {"$set": update})


def get_open_trade(symbol: str) -> dict | None:
    return trades_col.find_one({"symbol": symbol, "status": "open"})


# ══════════════════════════════════════════════════════════════════════════════
# Indicators
# ══════════════════════════════════════════════════════════════════════════════
def calc_atr(df: pd.DataFrame, length: int = 14) -> pd.Series:
    """True Range → SMA-smoothed ATR (mirrors TV SMA smoothing default)."""
    h, l, c = df["high"], df["low"], df["close"]
    prev_c  = c.shift(1)
    tr      = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
    return tr.rolling(length).mean()


def calc_rvol(df: pd.DataFrame, length: int = 20) -> float:
    """Relative volume of the last bar vs rolling average."""
    if len(df) < length + 1:
        return 1.0
    avg = df["volume"].iloc[-(length+1):-1].mean()
    if avg == 0:
        return 1.0
    return float(df["volume"].iloc[-1] / avg)


# ══════════════════════════════════════════════════════════════════════════════
# ORB State  (one instance per symbol)
# ══════════════════════════════════════════════════════════════════════════════
class ORBState:
    def __init__(self, symbol: str):
        self.symbol          = symbol
        self.date            = None       # current trading date (ET)
        self.orb_high        = None
        self.orb_low         = None
        self.orb_mid         = None
        self.orb_finalized   = False      # True once 9:45 bar is complete
        self.bull_fired      = False      # breakout signals fire once per day
        self.bear_fired      = False
        # Active trade state
        self.in_trade        = False
        self.side            = None       # "LONG" | "SHORT"
        self.entry_price     = None
        self.stop_level      = None
        self.target_level    = None
        self.initial_stop    = None
        self.moved_to_be     = False
        self.order_id        = None
        self.qty             = 0

    def reset_day(self, date):
        """Called at the start of each new trading day."""
        self.date          = date
        self.orb_high      = None
        self.orb_low       = None
        self.orb_mid       = None
        self.orb_finalized = False
        self.bull_fired    = False
        self.bear_fired    = False
        log.info(f"{self.symbol}: day reset for {date}")

    def update_orb(self, bar_high: float, bar_low: float):
        """Expand ORB range with a new bar inside the 9:30-9:45 window."""
        if self.orb_high is None:
            self.orb_high = bar_high
            self.orb_low  = bar_low
        else:
            self.orb_high = max(self.orb_high, bar_high)
            self.orb_low  = min(self.orb_low,  bar_low)
        self.orb_mid = (self.orb_high + self.orb_low) / 2

    def orb_ready(self) -> bool:
        return (self.orb_finalized
                and self.orb_high is not None
                and self.orb_low  is not None)


# ══════════════════════════════════════════════════════════════════════════════
# IBKR Async Client
# ══════════════════════════════════════════════════════════════════════════════
class IBKRClient:
    def __init__(self):
        self.ib = IB()

    async def connect(self):
        await self.ib.connectAsync(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
        log.info(f"IBKR connected | accounts: {self.ib.wrapper.accounts}")

    def disconnect(self):
        self.ib.disconnect()

    async def get_bars(self, symbol: str, duration: str = "2 D") -> pd.DataFrame | None:
        """Fetch 5-min OHLCV bars including pre/post market."""
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        for what in ("TRADES", "MIDPOINT"):
            raw = await self.ib.reqHistoricalDataAsync(
                contract,
                endDateTime="",
                durationStr=duration,
                barSizeSetting=BAR_SIZE,
                whatToShow=what,
                useRTH=False,
            )
            if raw:
                df = pd.DataFrame([{
                    "ts":     b.date,
                    "open":   b.open,
                    "high":   b.high,
                    "low":    b.low,
                    "close":  b.close,
                    "volume": b.volume,
                } for b in raw])
                df["ts"] = pd.to_datetime(df["ts"], utc=True).dt.tz_convert(ET)
                df.set_index("ts", inplace=True)
                log.debug(f"{symbol}: {len(df)} bars ({what})  last={df.index[-1]}")
                return df
        log.warning(f"{symbol}: no bars returned")
        return None

    async def get_last_price(self, symbol: str) -> float | None:
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        ticker = self.ib.reqMktData(contract, "106", False, False)
        await asyncio.sleep(3)
        self.ib.cancelMktData(contract)
        for val in [ticker.last, ticker.ask, ticker.close]:
            if val and float(val) > 0:
                return float(val)
        return None

    def has_position(self, symbol: str) -> bool:
        return any(
            p.contract.symbol == symbol and p.position != 0
            for p in self.ib.positions()
        )

    async def place_bracket(
        self, symbol: str, side: str,
        qty: int, entry: float, stop: float, target: float
    ):
        """Place a limit entry with stop-loss and take-profit bracket."""
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)

        action = "BUY" if side == "LONG" else "SELL"

        bracket = self.ib.bracketOrder(
            action, qty,
            limitPrice=round(entry, 2),
            takeProfitPrice=round(target, 2),
            stopLossPrice=round(stop, 2),
        )
        for o in bracket:
            o.outsideRth = True
            o.tif        = "DAY"

        trades = [self.ib.placeOrder(contract, o) for o in bracket]
        await asyncio.sleep(1)

        parent_id = trades[0].order.orderId
        log.info(
            f"BRACKET {side} {symbol} x{qty}  entry={entry:.2f}  "
            f"stop={stop:.2f}  target={target:.2f}  parentId={parent_id}"
        )
        return trades

    async def cancel_all_orders(self, symbol: str):
        for trade in self.ib.openTrades():
            if trade.contract.symbol == symbol:
                self.ib.cancelOrder(trade.order)
                log.info(f"Cancelled order {trade.order.orderId} for {symbol}")
        await asyncio.sleep(0.5)

    async def close_position(self, symbol: str, qty: int, side: str):
        """Market-close an open position."""
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        action = "SELL" if side == "LONG" else "BUY"
        order  = MarketOrder(action, abs(qty))
        order.outsideRth = True
        trade  = self.ib.placeOrder(contract, order)
        await asyncio.sleep(1)
        log.info(f"CLOSE {symbol} {action} x{abs(qty)}  orderId={trade.order.orderId}")
        return trade

    async def modify_stop(self, symbol: str, side: str,
                          qty: int, new_stop: float, old_stop_order_id: int):
        """Cancel existing stop and replace with breakeven stop."""
        await self.cancel_all_orders(symbol)
        await asyncio.sleep(0.3)
        contract = Stock(symbol, "SMART", "USD")
        await self.ib.qualifyContractsAsync(contract)
        action = "SELL" if side == "LONG" else "BUY"
        order  = StopOrder(action, qty, round(new_stop, 2))
        order.outsideRth = True
        order.tif        = "DAY"
        trade  = self.ib.placeOrder(contract, order)
        await asyncio.sleep(0.5)
        log.info(f"STOP MODIFIED {symbol} -> {new_stop:.2f}")
        return trade


# ══════════════════════════════════════════════════════════════════════════════
# Signal + Risk Logic
# ══════════════════════════════════════════════════════════════════════════════
def calc_stop_level(df: pd.DataFrame, side: str, state: ORBState,
                    last_close: float, atr: float) -> float:
    """Mirror TV's stop-loss calculation."""
    if SL_TYPE == "ATR":
        return last_close - atr * ATR_MULTI if side == "LONG" else last_close + atr * ATR_MULTI
    elif SL_TYPE == "Candle":
        idx = -(CANDLE_LENGTH + 1)
        return float(df["low"].iloc[idx]) if side == "LONG" else float(df["high"].iloc[idx])
    else:  # Midline
        return state.orb_mid


def passes_filters(now_et: datetime, side: str, rvol: float) -> bool:
    """Apply time, day, position-type, and RVOL filters."""
    if TIME_FILTER_START and TIME_FILTER_END:
        if not (TIME_FILTER_START <= now_et.time() <= TIME_FILTER_END):
            return False
    if ALLOWED_DAYS is not None:
        if now_et.weekday() not in ALLOWED_DAYS:
            return False
    if side == "LONG"  and not ALLOW_LONG:  return False
    if side == "SHORT" and not ALLOW_SHORT: return False
    if USE_RVOL and rvol < RVOL_MIN:
        log.info(f"RVOL filter failed: {rvol:.2f} < {RVOL_MIN}")
        return False
    return True


# ══════════════════════════════════════════════════════════════════════════════
# Per-Symbol Processing Loop
# ══════════════════════════════════════════════════════════════════════════════
async def process_symbol(symbol: str, state: ORBState, ibkr: IBKRClient):
    """Called every POLL_SECONDS. Implements the full ORB state machine."""
    now_et = datetime.now(ET)
    today  = now_et.date()

    # ── Day reset ────────────────────────────────────────────────────────────
    if state.date != today:
        state.reset_day(today)

    # ── Fetch bars ───────────────────────────────────────────────────────────
    df = await ibkr.get_bars(symbol)
    if df is None or len(df) < ATR_LENGTH + 2:
        log.warning(f"{symbol}: not enough bars")
        return

    # ── EOD close ────────────────────────────────────────────────────────────
    if now_et.time() >= EOD_CLOSE and state.in_trade:
        log.info(f"{symbol}: EOD — closing position")
        await ibkr.cancel_all_orders(symbol)
        await ibkr.close_position(symbol, state.qty, state.side)
        update_trade_db(symbol, {
            "status":      "closed",
            "exit_reason": "EOD",
            "exited_at":   datetime.now(timezone.utc),
        })
        state.in_trade = False
        return

    # Past cutoff — no new entries
    past_cutoff = now_et.time() >= TRADE_CUTOFF

    # ── Build ORB from bars in window ─────────────────────────────────────────
    orb_bars = df.between_time(ORB_START.strftime("%H:%M"),
                               ORB_END.strftime("%H:%M"))
    if not orb_bars.empty:
        for _, row in orb_bars.iterrows():
            state.update_orb(float(row["high"]), float(row["low"]))

    # Finalize ORB once we're past 9:45
    if now_et.time() >= ORB_END and not state.orb_finalized:
        if state.orb_high is not None:
            state.orb_finalized = True
            log.info(
                f"{symbol}: ORB finalized  high={state.orb_high:.2f}  "
                f"low={state.orb_low:.2f}  mid={state.orb_mid:.2f}"
            )

    if not state.orb_ready():
        log.debug(f"{symbol}: ORB not ready yet")
        return

    # ── Current bar values ───────────────────────────────────────────────────
    last_bar   = df.iloc[-1]
    prev_bar   = df.iloc[-2]
    last_close = float(last_bar["close"])
    prev_close = float(prev_bar["close"])

    atr_series = calc_atr(df, ATR_LENGTH)
    atr        = float(atr_series.iloc[-1])
    rvol       = calc_rvol(df, RVOL_LENGTH)

    log.info(
        f"{symbol}: close={last_close:.2f}  ORB[{state.orb_low:.2f}-{state.orb_high:.2f}]"
        f"  ATR={atr:.4f}  RVOL={rvol:.2f}"
    )

    # ── Breakeven check on open trade ────────────────────────────────────────
    if state.in_trade and BREAKEVEN_R > 0 and not state.moved_to_be:
        init_risk = abs(state.entry_price - state.initial_stop)
        profit    = (float(df["high"].iloc[-1]) - state.entry_price
                     if state.side == "LONG"
                     else state.entry_price - float(df["low"].iloc[-1]))

        if profit >= BREAKEVEN_R * init_risk:
            log.info(f"{symbol}: BREAKEVEN triggered at {state.entry_price:.2f}")
            state.moved_to_be = True
            state.stop_level  = state.entry_price
            await ibkr.modify_stop(symbol, state.side, state.qty,
                                   state.entry_price, -1)
            update_trade_db(symbol, {"stop_level":  state.entry_price,
                                     "moved_to_be": True})

    # ── No new entries if already in trade or past cutoff ────────────────────
    if state.in_trade or past_cutoff:
        return

    # ── Breakout detection ───────────────────────────────────────────────────
    bull_breakout = (
        not state.bull_fired
        and last_close > state.orb_high
        and prev_close <= state.orb_high
    )
    bear_breakout = (
        not state.bear_fired
        and last_close < state.orb_low
        and prev_close >= state.orb_low
    )

    if not bull_breakout and not bear_breakout:
        return

    side = "LONG" if bull_breakout else "SHORT"

    if bull_breakout:
        state.bull_fired = True
        log.info(f"{symbol}: BULL BREAKOUT  close={last_close:.2f} > ORB_HIGH={state.orb_high:.2f}")
    else:
        state.bear_fired = True
        log.info(f"{symbol}: BEAR BREAKOUT  close={last_close:.2f} < ORB_LOW={state.orb_low:.2f}")

    # ── Filters ──────────────────────────────────────────────────────────────
    if not passes_filters(now_et, side, rvol):
        log.info(f"{symbol}: breakout filtered out  side={side}")
        return

    # ── Stop and target calculation ───────────────────────────────────────────
    entry     = last_close
    stop      = calc_stop_level(df, side, state, last_close, atr)
    stop_dist = abs(entry - stop)

    if side == "LONG" and stop >= entry:
        log.warning(f"{symbol}: LONG stop {stop:.2f} >= entry {entry:.2f} — skipping")
        return
    if side == "SHORT" and stop <= entry:
        log.warning(f"{symbol}: SHORT stop {stop:.2f} <= entry {entry:.2f} — skipping")
        return

    target = (entry + RISK_REWARD * stop_dist if side == "LONG"
              else entry - RISK_REWARD * stop_dist)

    # ── Fixed quantity — always 10 shares ────────────────────────────────────
    qty = FIXED_QTY

    log.info(
        f"{symbol}: {side}  entry={entry:.2f}  stop={stop:.2f}  "
        f"target={target:.2f}  qty={qty}  (fixed)"
    )

    # ── Place bracket order ───────────────────────────────────────────────────
    trades = await ibkr.place_bracket(symbol, side, qty, entry, stop, target)

    # ── Update state ──────────────────────────────────────────────────────────
    state.in_trade     = True
    state.side         = side
    state.entry_price  = entry
    state.stop_level   = stop
    state.target_level = target
    state.initial_stop = stop
    state.moved_to_be  = False
    state.qty          = qty
    state.order_id     = trades[0].order.orderId

    # ── Persist to MongoDB ────────────────────────────────────────────────────
    save_trade_db({
        "symbol":        symbol,
        "side":          side,
        "entry_price":   entry,
        "stop_level":    stop,
        "target_level":  target,
        "initial_stop":  stop,
        "qty":           qty,
        "order_id":      state.order_id,
        "atr":           atr,
        "rvol":          rvol,
        "orb_high":      state.orb_high,
        "orb_low":       state.orb_low,
        "orb_mid":       state.orb_mid,
        "status":        "open",
        "moved_to_be":   False,
        "entered_at":    datetime.now(timezone.utc),
        "exit_price":    None,
        "exit_reason":   None,
        "exited_at":     None,
        "pnl_pct":       None,
    })

    signals_col.insert_one({
        "symbol":    symbol,
        "side":      side,
        "entry":     entry,
        "stop":      stop,
        "target":    target,
        "rvol":      rvol,
        "qty":       qty,
        "timestamp": datetime.now(timezone.utc),
    })


# ══════════════════════════════════════════════════════════════════════════════
# Main Event Loop
# ══════════════════════════════════════════════════════════════════════════════
async def main():
    log.info("=== ORB Bot Starting ===")
    log.info(f"Symbols: {SYMBOLS}  |  Bar: {BAR_SIZE}  |  Poll: {POLL_SECONDS}s")
    log.info(f"ORB window: {ORB_START}-{ORB_END} ET  |  Cutoff: {TRADE_CUTOFF} ET")
    log.info(f"SL: {SL_TYPE}  RR: {RISK_REWARD}  BE@{BREAKEVEN_R}R  Qty: {FIXED_QTY} (fixed)")

    util.patchAsyncio()

    ibkr   = IBKRClient()
    await ibkr.connect()

    states = {sym: ORBState(sym) for sym in SYMBOLS}

    try:
        while True:
            for symbol in SYMBOLS:
                try:
                    await process_symbol(symbol, states[symbol], ibkr)
                except Exception as e:
                    log.error(f"{symbol}: error in process_symbol — {e}", exc_info=True)

            log.info(f"Sleeping {POLL_SECONDS}s...")
            await asyncio.sleep(POLL_SECONDS)

    except KeyboardInterrupt:
        log.info("ORB Bot stopped by user.")
    finally:
        for symbol in SYMBOLS:
            state = states[symbol]
            if state.in_trade:
                log.info(f"{symbol}: shutdown — closing position")
                try:
                    await ibkr.cancel_all_orders(symbol)
                    await ibkr.close_position(symbol, state.qty, state.side)
                except Exception as e:
                    log.error(f"{symbol}: shutdown close error — {e}")
        ibkr.disconnect()
        log.info("IBKR disconnected.")


if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    if loop.is_running():
        loop.create_task(main())
    else:
        util.run(main())


2026-04-03 07:46:05,592 [INFO] === ORB Bot Starting ===
2026-04-03 07:46:05,595 [INFO] Symbols: ['SPY', 'QQQ']  |  Bar: 5 mins  |  Poll: 60s
2026-04-03 07:46:05,596 [INFO] ORB window: 09:30:00-09:45:00 ET  |  Cutoff: 11:00:00 ET
2026-04-03 07:46:05,597 [INFO] SL: ATR  RR: 2.0  BE@1.0R  Qty: 10 (fixed)
2026-04-03 07:46:05,947 [INFO] IBKR connected | accounts: ['DUD087866']
2026-04-03 07:46:05,948 [INFO] SPY: day reset for 2026-04-03
2026-04-03 07:46:06,262 [INFO] QQQ: day reset for 2026-04-03
2026-04-03 07:46:06,724 [INFO] Sleeping 60s...
2026-04-03 07:47:07,180 [INFO] Sleeping 60s...
